# 03. Multi-output LightGBM 학습 및 예측 (Target-Centric)

## 📋 개요
이 노트북은 **타깃 중심 정렬(Target-Centric Alignment)** 방식을 사용하여 모델을 학습합니다.
- **핵심 논리**: $t$ 시점의 가격을 맞추기 위해 $t-i$ 시점의 데이터를 참조하는 Direct Forecasting 전략을 사용합니다.
- **데이터 정렬**: 예측 결과의 날짜(`date`)가 실제 예측 대상일과 일치하도록 구성하여 분석의 직관성을 높였습니다.
- **하이브리드 구조**: 5일치(1주일) Chunk를 한 번에 예측하는 Multi-output 모델을 구축합니다.

## ✨ H1+H2 패치 적용 (2026-02-08)
- **경로 관리 중앙화**: `ProjectPaths` 클래스 사용
- **폴더 구조 개선**: `03_training/` 독립 디렉토리

In [ ]:
# ============================================================
# 03_train_predict.ipynb — 패치 모음 (v3.8.0)
#
# [imports 셀]   is_ensemble 추가 import
# [config 셀]    앙상블 조합 지정 시 조기 차단
# [init_objects 셀]  약칭(lgbm/rf) 허용, MLP 분기, 방어 코드 정리
# ============================================================

## 🔧 Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
import copy
from tqdm import tqdm

from src.utils.config import load_config, ProjectPaths, is_ensemble
from src.modeling.trainer import WalkForwardTrainer
from src.models.artifact import save_model_artifact

warnings.filterwarnings('ignore')

In [ ]:
# ── [config 셀] ──────────────────────────────────────────────
cfg       = load_config()
train_cfg = cfg['training']

active_model_str = cfg.get('active_model', 'lightgbm')
if is_ensemble(active_model_str):
    raise ValueError(
        f"active_model='{active_model_str}'은 앙상블 조합입니다.\n"
        f"03단계는 단일 모델 전용입니다. 앙상블은 03b를 사용하세요."
    )

paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

print(f"🚀 [Step 3] Multi-output 학습 시작")
print(f"   - 기준일 : {paths.reference_date}")
print(f"   - 모델   : {paths.folder_name}")
print(f"\n📂 사용 경로:")
print(f"   - 입력: {paths.get_dataset_parquet()}")
print(f"   - 모델: {paths.get_model_dir()}")

## 1️⃣ 데이터 로드
02단계에서 생성된 통합 Feature 데이터셋을 로드합니다. 개별 시점의 피처 시프트는 Trainer 내부에서 타깃별로 수행됩니다.

In [ ]:
# ==========================================
# ✨ H2 패치: ProjectPaths 메서드 사용
# ==========================================

print("📥 Loading dataset...")
df = pd.read_parquet(paths.get_dataset_parquet())
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

feature_cols = [c for c in df.columns if c.startswith('feature_') or c in ['risk_composite']]

print(f"   - 학습 데이터 행수: {len(df):,}")
print(f"   - 사용 피처 수: {len(feature_cols)}")

## 2️⃣ 모델 및 Trainer 초기화
Multi-output을 지원하는 `LightGBMModel`을 생성하고, 타깃 중심 정렬 로직이 포함된 `WalkForwardTrainer`를 설정합니다.

In [ ]:
# ============================================================
# 03_train_predict.ipynb — init_objects 셀 패치 (v3.8.0)
#
# 변경 내역:
#   - MLP 분기 추가 (model_type == "mlp")
#   - 기존 else → "ensemble" 전용 분기로 분리
#     (ensemble은 03b에서 처리하므로 여기서 직접 선택 시 안내)
#   - active_model 주석 업데이트
# ============================================================

model_type = active_model_str  # config 셀에서 단일 모델 검증 완료

if model_type in ("lightgbm", "lgbm"):
    print("🤖 Initializing LightGBM Model...")
    from src.models.lightgbm_model import LightGBMModel
    model = LightGBMModel(
        model_version=f"v1_lgbm_{paths.reference_date}",
        params=train_cfg['lgbm_params'],
        feature_list=feature_cols,
        categorical_features=[]
    )
    fit_kwargs = {
        'num_boost_round': 1000,
        'callbacks': [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0)
        ]
    }

elif model_type in ("randomforest", "rf"):
    print("🌲 Initializing Random Forest Multi-output Model...")
    from src.models.randomforest_model import RandomForestMultiModel
    model = RandomForestMultiModel(
        model_version=f"v1_rf_{paths.reference_date}",
        params=train_cfg["randomforest_params"],
        feature_list=feature_cols
    )
    fit_kwargs = {}

elif model_type == "mlp":
    print("🧠 Initializing MLP Multi-output Model...")
    from src.models.mlp_model import MLPModel
    model = MLPModel(
        model_version=f"v1_mlp_{paths.reference_date}",
        params=train_cfg["mlp_params"],
        feature_list=feature_cols
    )
    fit_kwargs = {
        'epochs':   train_cfg["mlp_params"].get("epochs", 200),
        'patience': train_cfg["mlp_params"].get("patience", 15),
    }

else:
    # config.py의 resolve_model_name으로 최종 검증 (명확한 오류 메시지 확보)
    from src.utils.config import resolve_model_name
    resolve_model_name(model_type)   # 알 수 없는 이름이면 ValueError

horizons    = train_cfg.get('horizons', [1, 2, 3, 4, 5])
target_base = train_cfg.get('target_col_name', 'target_log_close')

model.target_columns = [f"{target_base}_h{h}" for h in horizons]

print("🔧 Initializing Trainer...")
trainer = WalkForwardTrainer(
    model=model,
    feature_cols=feature_cols,
    target_col_name=train_cfg.get('target_col_name', 'target_log_close'),
    target_type=train_cfg.get('target_type', 'log_close'),
    horizons=horizons,
    date_col='date'
)

## 3️⃣ Walk-Forward 학습 실행
각 시점($t$)의 가격을 정답으로 두고, $t-1, t-2, \dots, t-5$의 피처를 각각 매칭하여 5개의 내부 모델을 학습합니다.

In [ ]:
print("🏃 Running Target-Centric Walk-Forward Training...")

results = trainer.run(
    df=df,
    train_end=train_cfg['train_end'],
    valid_window_days=train_cfg['valid_window_days'],
    test_window_days=train_cfg['test_window_days'],
    fit_kwargs=fit_kwargs
)

## 4️⃣ 결과 저장 및 성능 요약

In [ ]:
# ==========================================
# ✨ H1+H2 패치: ProjectPaths 메서드 사용
# ==========================================
# 03_train_predict.ipynb — 저장 셀 변경 사항 (v3.5.0)
#
# 기존: results['test_predictions'] → predictions.parquet (단일 파일)
# 변경: val_predictions  → val_predictions.parquet  (앙상블 가중치 최적화용)
#       test_predictions → test_predictions.parquet (최종 평가 전용)

print("\n💾 Saving predictions & model artifact...")

val_pred_df  = results['val_predictions']
test_pred_df = results['test_predictions']
target_type  = results['target_type']   # "log_return" or "log_close"

# ==========================================
# 예측값 → 실제 가격 역산 (v3.9.0)
# ==========================================
#
# [log_close 모드] (기존, 유지)
#   pred_target_log_close_h{n}(t) = log(close(t+n))
#   → pred_close_h{n} = exp(pred_target_log_close_h{n})
#
# [log_return_1d 모드] (v3.9.0 신규)
#   pred_target_log_return_1d_h{n}(t) = log1p(change_pct(t+n))
#   → close(t+h) = close(t) × exp( Σ pred_log_return_1d_h{i}, i=1..h )
#   cumsum 역산: h번째 시점까지 예측 등락률을 누적합산
#   기준가(close(t))는 date+ticker 키로 원본 df에서 조인

if target_type == "log_return_1d":
    # t 시점의 실제 close 기준값 조인
    close_ref = df[['date', 'ticker', 'close']].drop_duplicates()

    for df_pred in [val_pred_df, test_pred_df]:
        df_merged = df_pred.merge(close_ref, on=['date', 'ticker'], how='left')
        log_close_base = np.log(df_merged['close'].clip(lower=1e-9))

        # horizon별 pred 컬럼명 리스트 (h1, h2, ... 순 보장)
        sorted_target_cols = sorted(
            results['target_cols'],
            key=lambda c: int(c.split('_h')[-1])
        )

        for idx, col in enumerate(sorted_target_cols):
            pred_col = f'pred_{col}'                     # pred_target_log_return_1d_h{n}
            if pred_col not in df_merged.columns:
                continue

            # cumsum: h1~h{n}까지의 예측 log 등락률 누적
            cum_log_return = sum(
                df_merged[f'pred_{c}']
                for c in sorted_target_cols[:idx + 1]
            )
            pred_log_close = log_close_base + cum_log_return
            df_merged[f'pred_log_close_{col}'] = pred_log_close
            df_merged[f'pred_close_{col}']     = np.exp(pred_log_close)

        # merge 결과로 교체
        if df_pred is val_pred_df:
            val_pred_df = df_merged
        else:
            test_pred_df = df_merged

else:  # log_close 모드
    for df_pred in [val_pred_df, test_pred_df]:
        for col in results['target_cols']:
            pred_col = f'pred_{col}'
            if pred_col in df_pred.columns:
                df_pred[f'pred_close_{col}'] = np.exp(df_pred[pred_col])

# ──────────────────────────────────────
# Parquet 저장
# ──────────────────────────────────────
val_parquet_path  = paths.get_model_dir() / "val_predictions.parquet"
test_parquet_path = paths.get_model_dir() / "test_predictions.parquet"

val_pred_df.to_parquet(val_parquet_path,  index=False)
test_pred_df.to_parquet(test_parquet_path, index=False)

print(f"   - [Val]  Saved: {val_parquet_path}")
print(f"   - [Test] Saved: {test_parquet_path}")

# ──────────────────────────────────────
# 성능 요약
# ──────────────────────────────────────
print(f"\n📊 성능 요약  (target_type={target_type})")
print(f"   검증  Avg RMSE : {results['valid_metrics']['avg_rmse']:.6f}")
print(f"   테스트 Avg RMSE: {results['test_metrics']['avg_rmse']:.6f}")
print(f"\n   Horizon별 테스트 RMSE:")
for col, metrics in results['test_metrics']['per_horizon'].items():
    rmse = metrics['rmse']
    ic = metrics['ic_mean']
    print(f"     {col}: RMSE={rmse:.6f}, IC={ic:.4f}")

# 종목별 개별 CSV 저장 (디버깅/사람용)
csv_pred_dir = paths.get_predictions_csv_dir()
csv_pred_dir.mkdir(exist_ok=True)
print(f"   - [Individual] Saving ticker CSVs to {csv_pred_dir}...")

# ticker_name_map 로드 (01단계 master 활용)
try:
    df_master = pd.read_csv(paths.get_ticker_master())
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
except Exception:
    ticker_name_map = {}

for ticker, group in tqdm(test_pred_df.groupby('ticker'), desc="Saving CSVs"):
    name = ticker_name_map.get(str(ticker), f"ticker_{ticker}")
    safe_name = str(name).replace('/', '_').replace('\\', '_')
    group.to_csv(csv_pred_dir / f"{safe_name}.csv", index=False, encoding='utf-8-sig')

# 모델 아티팩트 저장
model_save_dir = paths.get_model_dir()

save_model_artifact(
    model_name=model_type,
    model_version=model.model_version,
    model_object=results['final_model'],
    metadata={
        "test_metrics": results['test_metrics'],
        "target_columns": model.target_columns
    },
    model_dir=model_save_dir
)

print("\n✅ [Step 3] 모든 산출물 저장 완료")
print(f"\n📂 저장 위치:")
print(f"   - 모델: {model_save_dir}")
print(f"   - 개별 CSV: {csv_pred_dir}")

display(test_pred_df.head())

## 🏁 모델 학습 완료
- **예측 데이터**: `predictions.parquet`에서 각 날짜별로 $t-1 \sim t-5$ 시점에 예측한 값들을 확인할 수 있습니다.
- **다음 단계**: 생성된 Chunk 단위 예측값을 기반으로 60일까지의 **재귀적 확장(Recursive Extension)** 및 전략 백테스트를 수행하십시오.

## ✨ H1+H2 패치 적용 완료
- 경로 관리가 `ProjectPaths` 클래스로 중앙화되었습니다.
- 결과물은 `data/03_training/{ref_date}/` 에 저장됩니다.